In [ ]:
# Python code for confirming the Lipschitz Constant of the SLL model

In [1]:
# Torch Library
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.backends.cudnn as cudnn
from torch import optim
from torch import autograd
from torch.autograd import Variable
from torch.autograd import Function
from torch.utils.data import DataLoader
from torch.utils.data import random_split
from torch.utils.data.sampler import SubsetRandomSampler
from torch import linalg as LA
from torch.autograd import Variable
from torch.autograd import Function

# Torchvision Library
import torchvision
from torchvision import transforms
from torchvision import datasets
from torchvision.datasets import CIFAR10

# Common Library
import os
import tarfile
import numpy as np
from os.path import exists, realpath


In [8]:
# SLL Library 
# (SLL model from paper "A Unified Algebraic Perspective on Lipschitz Neural Networks")
# https://github.com/araujoalexandre/Lipschitz-SLL-Networks
# Araujo, Alexandre, et al. "A unified algebraic perspective on lipschitz neural networks." 
# arXiv preprint arXiv:2303.03169 (2023).

from SLL.models.model import NormalizedModel
from SLL.models.model import LipschitzNetwork
from SLL.data.readers import readers_config
from SLL.models.layers import SDPBasedLipschitzConvLayer
from SLL.models.layers import SDPBasedLipschitzLinearLayer


In [15]:
# SLLsmall config

class Config:

  def __init__(self, dataset, size):
    self.mode = 'certified'
    self.dataset = dataset
    self.ngpus = 4
    self.batch_size = 500
    self.shift_data = True

    config = self.set_archi(20, 7, 45, 2048)

  def set_archi(self, n_conv, n_dense, w, dense_inner_dim):
    self.n_conv = n_conv
    self.n_dense = n_dense
    self.conv_inner_dim = 5
    self.dense_inner_dim = dense_inner_dim
    self.w = w 

# SLLsmall data import

config = Config('cifar10', 'small')
Reader = readers_config['cifar10']
reader = Reader(config, 256, False, is_training=False)
data_loader, _ = reader.load_dataset()

# SLLsmall model import

modelsmall = LipschitzNetwork(config, reader.n_classes)
modelsmall = NormalizedModel(modelsmall, reader.means, reader.stds)
modelsmall = torch.nn.DataParallel(modelsmall)
modelsmall.load_state_dict(torch.load('SLL/modelsmall.ckpt')['model_state_dict'])
modelsmall = modelsmall.cuda()

In [16]:
# SLLsmall model import

paths = './SLL/modelsmall.ckpt'
data = torch.load(paths)['model_state_dict']


In [17]:
# Generic power method to determine Global Lipschitz

def _norm_gradient_sq(linear_fun, v):
    v = Variable(v, requires_grad=True)
    loss = torch.norm(linear_fun(v))**2
    loss.backward()
    return v.grad.data


def generic_power_method(affine_fun, input_size, eps=1e-8, max_iter=500, use_cuda=False):
    zeros = torch.zeros(input_size)
    if use_cuda:
        zeros = zeros.cuda()
    bias = affine_fun(Variable(zeros))
    linear_fun = lambda x: affine_fun(x) - bias

    def norm(x, p=2):
      """ Norm for each batch
      """
      norms = Variable(torch.zeros(x.shape[0]))
      if use_cuda:
          norms = norms.cuda()
      for i in range(x.shape[0]):
          norms[i] = x[i].norm(p=p)
      return norms

    # Initialise with random values
    v = torch.randn(input_size)
    v = F.normalize(v.view(v.shape[0], -1), p=2, dim=1).view(input_size)
    if use_cuda:
        v = v.cuda()

    stop_criterion = False
    it = 0
    while not stop_criterion:
        previous = v
        v = _norm_gradient_sq(linear_fun, v)
        v = F.normalize(v.view(v.shape[0], -1), p=2, dim=1).view(input_size)
        stop_criterion = (torch.norm(v - previous) < eps) or (it > max_iter)
        it += 1
    # Compute Rayleigh product to get eivenvalue
    u = linear_fun(Variable(v))  # unormalized left singular vector
    eigenvalue = norm(u)
    u = u.div(eigenvalue)
    return eigenvalue.item()

In [18]:
# If our Lipschitz is smaller than 1, we use 1. 

Global_lip = 1.0

In [19]:
# Determine the global Lipschitz for SLL small

n_conv_layers = len(list(filter(
    lambda x: 'module.model.model' in x and 'kernel' in x, data.keys())))
n_dense_layers = len(list(filter(
    lambda x: 'module.model.model' in x and 'weight' in x, data.keys())))

for i in range(1, n_conv_layers+1):
    layer_ckpt = {
        'kernel': data[f'module.model.model.{i}.kernel'],
        'bias': data[f'module.model.model.{i}.bias'],
        'q': data[f'module.model.model.{i}.q']
    }
    cout, cin = data[f'module.model.model.{i}.kernel'].shape[:2]
    layer = SDPBasedLipschitzConvLayer(cin, cout)
    layer.load_state_dict(layer_ckpt)
    layer = layer.cuda()
    layer.kernel.requires_grad = False
    layer.bias.requires_grad = False
    layer.q.requires_grad = False

    input_size = (1, cin, 32, 32)
    sv = generic_power_method(layer, input_size, eps=1e-8, max_iter=500, use_cuda=True)
    print(f'SLL Conv {i}: {sv}')

for i in range(n_conv_layers+3, n_dense_layers+n_conv_layers+2):
    layer_ckpt = {
        'weight': data[f'module.model.model.{i}.weight'],
        'bias': data[f'module.model.model.{i}.bias'],
        'q': data[f'module.model.model.{i}.q'],
    }
    cout, cin = data[f'module.model.model.{i}.weight'].shape
    layer = SDPBasedLipschitzLinearLayer(cin, cout)
    layer.load_state_dict(layer_ckpt)
    layer = layer.cuda()
    layer.weight.requires_grad = False
    layer.bias.requires_grad = False
    layer.q.requires_grad = False
    
    input_size = (1, cout, cin)
    sv = generic_power_method(layer, input_size, eps=1e-8, max_iter=500, use_cuda=True)
    print(f'SLL Dense {i}: {sv}')

SLL Conv 1: 0.9999998807907104
SLL Conv 2: 0.9999996423721313
SLL Conv 3: 0.9999999403953552
SLL Conv 4: 1.0
SLL Conv 5: 0.9999998807907104
SLL Conv 6: 0.9999997615814209
SLL Conv 7: 0.9999999403953552
SLL Conv 8: 1.0
SLL Conv 9: 1.0
SLL Conv 10: 1.0
SLL Conv 11: 0.9999999403953552
SLL Conv 12: 0.9999999403953552
SLL Conv 13: 0.9999999403953552
SLL Conv 14: 0.9999999403953552
SLL Conv 15: 1.0
SLL Conv 16: 0.9999964237213135
SLL Conv 17: 0.9999999403953552
SLL Conv 18: 1.0
SLL Conv 19: 1.0
SLL Conv 20: 1.0
SLL Dense 23: 0.9999992847442627
SLL Dense 24: 0.999993622303009
SLL Dense 25: 0.9999962449073792
SLL Dense 26: 0.9999942779541016
SLL Dense 27: 0.9999924898147583
SLL Dense 28: 0.9999909996986389
SLL Dense 29: 0.9999848008155823
